# Khám phá dữ liệu RAG tiếng Việt

**[TÁI SỬ DỤNG]** Notebook chỉ đọc `data/`, không sửa dữ liệu, không huấn luyện và không dùng GPU. Chạy từ terminal bằng `bash colab_run.sh eda` hoặc từ môi trường local `uv sync --locked --extra eda` (xem README). Các thống kê mô tả cả dev/test, nhưng nội dung câu hỏi và gom cụm chỉ dùng **dev**, không dùng test để chọn cấu hình.

In [1]:
from collections import Counter
from pathlib import Path
import json
import os

from datasets import load_from_disk
from huggingface_hub import hf_hub_download
from IPython.display import display
import pandas as pd
from tokenizers import Tokenizer

from ragvi.eda import token_lengths, wording_groups

pd.set_option('display.max_colwidth', None)
DATA = Path(os.environ.get('RAGVI_DATA_DIR', 'data'))
if not DATA.is_dir() and not DATA.is_absolute():
    DATA = Path('..') / DATA
if not (DATA / 'processed').is_dir():
    raise FileNotFoundError(f'Không tìm thấy dữ liệu đã xử lý: {DATA.resolve()}')

def read_jsonl(path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f]


## 1. Định dạng và cột dữ liệu [TÁI SỬ DỤNG]

- `raw/` và `processed/`: Hugging Face `save_to_disk`. **Arrow** chứa dữ liệu theo cột, `state.json` và `dataset_info.json` mô tả bộ dữ liệu, `dataset_dict.json` liệt kê split. Các Arrow `cache-*` không phải split mới.
- `eval/`: **JSONL**, mỗi dòng là một câu hỏi hoặc đoạn văn. `manifest.json` ghi nguồn/seed và mã băm, `removed_doc_ids.json` liệt kê điều luật loại khỏi kho Zalo.
- ViQuAD: câu hỏi, đoạn, đáp án; Zalo: `corpus` (điều luật), `queries` (truy vấn), `qrels` (quan hệ truy vấn–điều luật, không phải đáp án viết sẵn).

In [2]:
inventory = {}
for path in DATA.rglob('*'):
    if path.is_file() and path != DATA / 'eval/zalo/reserve.jsonl':
        key = (path.relative_to(DATA).parts[0], path.suffix)
        count, size = inventory.get(key, (0, 0))
        inventory[key] = (count + 1, size + path.stat().st_size)
display(pd.DataFrame([(stage, fmt, count, round(size / 1e6, 3))
                      for (stage, fmt), (count, size) in sorted(inventory.items())],
                     columns=['thư mục', 'đuôi tệp', 'số tệp', 'MB']))

arrow = []
for path in sorted((DATA / 'processed').rglob('dataset_info.json')):
    features = json.loads(path.read_text(encoding='utf-8'))['features']
    cols = ', '.join(f'{name}: {spec.get("dtype", spec.get("_type", "struct"))}'
                     for name, spec in features.items())
    arrow.append((path.parent.relative_to(DATA).as_posix(), cols))
display(pd.DataFrame(arrow, columns=['bộ Arrow', 'cột']))


,thư mục,đuôi tệp,số tệp,MB
0,eval,.json,3,0.003
1,eval,.jsonl,5,6.356
2,processed,.arrow,6,164.934
3,processed,.json,13,0.009
4,raw,.arrow,11,329.808
5,raw,.json,13,0.009


,bộ Arrow,cột
0,processed/viquad/test,"id: string, uit_id: string, title: string, context: string, question: string, answers: struct, is_impossible: bool, plausible_answers: struct"
1,processed/viquad/train,"id: string, uit_id: string, title: string, context: string, question: string, answers: struct, is_impossible: bool, plausible_answers: struct"
2,processed/viquad/validation,"id: string, uit_id: string, title: string, context: string, question: string, answers: struct, is_impossible: bool, plausible_answers: struct"
3,processed/zalo_legal/corpus,"id: string, title: string, text: string"
4,processed/zalo_legal/qrels,"query-id: string, corpus-id: string, score: int64"
5,processed/zalo_legal/queries,"id: string, text: string"


## 2. Câu hỏi, nhãn và bằng chứng [TÁI SỬ DỤNG]

Tách **test nguồn ViQuAD** (không có đáp án vàng) khỏi **test của nghiên cứu** (lấy từ train/validation). Trên Zalo, không trả lời được nghĩa là các điều luật vàng bị loại khỏi kho dùng để truy xuất, không đồng nghĩa toàn bộ văn bản luật không có câu trả lời. `group` Zalo nối truy vấn dùng chung điều luật vàng, khác cụm câu chữ ở mục 4.

In [3]:
viquad = load_from_disk(str(DATA / 'processed/viquad'))
zalo = {part: load_from_disk(str(DATA / 'processed/zalo_legal' / part))
        for part in ('corpus', 'queries', 'qrels')}
sources = []
for split, ds in viquad.items():
    missing = sum(ans is None for ans in ds['answers'])
    impossible = sum(ds['is_impossible']) if split != 'test' else None
    sources.append((f'ViQuAD / {split}', len(ds), missing, impossible))
display(pd.DataFrame(sources, columns=['split nguồn', 'số dòng', 'đáp án null', 'nhãn không trả lời được']))
print(f'Zalo: {len(zalo["corpus"]):,} điều, {sum(not text.strip() for text in zalo["corpus"]["text"]):,} thân điều rỗng; '
      f'{len(zalo["queries"]):,} dòng truy vấn, {len(set(zalo["queries"]["id"])):,} mã duy nhất; '
      f'{len(zalo["qrels"]):,} dòng qrels.')

eval_sets = {}
for dataset, group_key in [('viquad', 'title'), ('zalo', 'group')]:
    for split in ('dev', 'test'):
        eval_sets[dataset, split] = read_jsonl(DATA / 'eval' / dataset / f'{split}.jsonl')
    dev, test = (eval_sets[dataset, s] for s in ('dev', 'test'))
    assert not {r['qid'] for r in dev} & {r['qid'] for r in test}, f'Trùng ID dev/test: {dataset}'
    assert not {r[group_key] for r in dev} & {r[group_key] for r in test}, f'Trùng nhóm dev/test: {dataset}'

summary = []
for (dataset, split), rows in eval_sets.items():
    group_key = 'title' if dataset == 'viquad' else 'group'
    counts = Counter(r['answerable'] for r in rows)
    summary.append((dataset, split, len(rows), counts[True], counts[False],
                    len({r[group_key] for r in rows}), ', '.join(rows[0])))
display(pd.DataFrame(summary, columns=['bộ', 'split', 'số câu', 'trả lời được',
                                      'không trả lời được', 'số bài/nhóm vàng', 'cột JSONL']))


,split nguồn,số dòng,đáp án null,nhãn không trả lời được
0,ViQuAD / train,28454,0,9216.0
1,ViQuAD / validation,3814,0,1161.0
2,ViQuAD / test,7301,7301,NaN


Zalo: 61,425 điều, 357 thân điều rỗng; 818 dòng truy vấn, 788 mã duy nhất; 793 dòng qrels.


,bộ,split,số câu,trả lời được,không trả lời được,số bài/nhóm vàng,cột JSONL
0,viquad,dev,100,77,23,3,"qid, question, answerable, gold_doc_id, title, answers"
1,viquad,test,400,280,120,118,"qid, question, answerable, gold_doc_id, title, answers"
2,zalo,dev,100,100,0,19,"qid, question, answerable, gold_doc_ids, group"
3,zalo,test,400,280,120,235,"qid, question, answerable, gold_doc_ids, group"


## 3. Số token và đuôi phân phối [TÁI SỬ DỤNG]

Đếm toàn bộ dòng bằng tokenizer tham chiếu **BAAI/bge-m3** theo D7; không thêm token đặc biệt, không cắt và không đếm token của prompt/GGUF. Báo cáo số vượt 512 để thấy tài liệu nào dễ bị cắt nếu nhúng nguyên đoạn. ViQuAD dùng đoạn trong `corpus.jsonl`; Zalo đếm `title + text` cho toàn kho đã làm sạch trước khi loại điều luật. Không cộng kho và các split để gọi là một cỡ dữ liệu: đó là các đơn vị khác nhau.

In [4]:
REV = '5617a9f61b028005a4858fdac845db406aefb181'
tokenizer = Tokenizer.from_file(hf_hub_download('BAAI/bge-m3', 'tokenizer.json', revision=REV))
tokenizer.no_truncation()
tokenizer.no_padding()
viquad_corpus = read_jsonl(DATA / 'eval/viquad/corpus.jsonl')
texts = {
    'ViQuAD / đoạn trong kho': (r['text'] for r in viquad_corpus),
    'Zalo / điều trong kho': (r['title'] + '\n' + r['text'] for r in zalo['corpus']),
}
for split, ds in viquad.items():
    texts[f'ViQuAD / câu hỏi nguồn {split}'] = ds['question']
texts['Zalo / truy vấn nguồn (mã duy nhất)'] = dict(zip(zalo['queries']['id'], zalo['queries']['text'])).values()
for (name, split), rows in eval_sets.items():
    texts[f'{name} / câu hỏi đánh giá {split}'] = [r['question'] for r in rows]

stats = []
for source, lines in texts.items():
    lengths = pd.Series(token_lengths(lines, tokenizer), dtype='int64')
    stats.append((source, len(lengths), int(lengths.sum()), round(lengths.median(), 1),
                  round(lengths.quantile(.95), 1), int(lengths.max()), int((lengths > 512).sum()),
                  round(100 * (lengths > 512).mean(), 2)))
display(pd.DataFrame(stats, columns=['nguồn', 'số dòng', 'tổng token', 'p50', 'p95', 'max', '>512', '% >512']))


,nguồn,số dòng,tổng token,p50,p95,max,>512,% >512
0,ViQuAD / đoạn trong kho,5311,1135938,191.0,369.5,2091,41,0.77
1,Zalo / điều trong kho,61425,21075880,223.0,945.8,67395,10719,17.45
2,ViQuAD / câu hỏi nguồn train,28454,482640,16.0,27.0,66,0,0.00
3,ViQuAD / câu hỏi nguồn validation,3814,63695,16.0,26.3,75,0,0.00
4,ViQuAD / câu hỏi nguồn test,7301,122614,16.0,26.0,42,0,0.00
5,Zalo / truy vấn nguồn (mã duy nhất),788,16705,21.0,33.0,40,0,0.00
6,viquad / câu hỏi đánh giá dev,100,1528,15.0,22.0,30,0,0.00
7,viquad / câu hỏi đánh giá test,400,6901,17.0,27.0,37,0,0.00
8,zalo / câu hỏi đánh giá dev,100,2015,20.0,32.0,38,0,0.00
9,zalo / câu hỏi đánh giá test,400,8627,21.0,34.0,38,0,0.00


## 4. Cụm câu hỏi gần nhau và ví dụ [TÁI SỬ DỤNG]

Dùng TF-IDF n-gram ký tự (3–5), chuẩn hóa Unicode/case, giữ nguyên dấu tiếng Việt, nối cặp có cosine ≥ 0,60. Đây là nhóm **gần câu chữ**, không phải nhãn cùng ý nghĩa hay cùng đáp án: các câu có thể nối bắc cầu, và một phủ định hoặc số Điều khác có thể đổi ý. Các nhóm chỉ để đọc lỗi trên **dev**, không áp vào cách chia tập.

In [5]:
wiki_by_id = {r['doc_id']: r['text'] for r in viquad_corpus}
legal_index = {doc_id: i for i, doc_id in enumerate(zalo['corpus']['id'])}
for dataset in ('viquad', 'zalo'):
    rows = eval_sets[dataset, 'dev']
    groups = wording_groups([r['question'] for r in rows])
    print(f'\n{dataset}: {len(groups)} cụm, {sum(map(len, groups))}/{len(rows)} câu trong cụm.')
    for group in groups[:4]:
        print(f'  Cụm {len(group)} câu:')
        for i in group[:3]:
            print('   -', rows[i]['question'])
    print('  Ví dụ gắn bằng chứng:')
    for label in (True, False):
        row = next((r for r in rows if r['answerable'] == label), None)
        if row is None:
            continue
        print(f'   - {row["qid"]} | trả lời được={label} | {row["question"]}')
        if dataset == 'viquad':
            print('     Đáp án vàng:', [a['text'] for a in row['answers']])
            print('     Đoạn nguồn:', wiki_by_id[row['gold_doc_id']][:200].replace('\n', ' '))
        else:
            for doc_id in row['gold_doc_ids']:
                article = zalo['corpus'][legal_index[doc_id]]
                print('     Điều luật vàng:', doc_id, article['title'])
                print('     Trích đoạn:', article['text'][:200].replace('\n', ' '))



viquad: 1 cụm, 2/100 câu trong cụm.
zalo: 9 cụm, 24/100 câu trong cụm.
Các ví dụ gốc không được lưu trong bản trên Git.


## Đọc kết quả để chọn việc tiếp theo

- So sánh độ dài đuôi giữa ViQuAD và Zalo: điều luật dài có thể mất bằng chứng nếu đưa nguyên văn vào retriever 512 token; xem kết quả theo dải độ dài thay vì chỉ một trung bình.
- ViQuAD dev chỉ phủ ít bài hơn test: coi nó là bộ kiểm tra triển khai, không suy ra độ cải thiện trên test từ vài ví dụ.
- Dùng các câu gần nhau trên dev để đọc lỗi xếp hạng, nhất là câu khác số Điều/thực thể. Giữ tập test đóng băng để đo `H_gold` và độ đúng sau khi chốt cấu hình.